# HI 5304 — APIs & JSON in Python (Jupyter Notebook)

*Created: 2026-01-25*

This notebook teaches the essentials of working with **APIs** and **JSON** in Python using **Jupyter** (ideal for GitHub Codespaces). It is **read-only** and uses public endpoints.


## Learning goals
- Make a GET request to a public API
- Understand JSON → Python dictionaries/lists
- Handle parameters, headers, and common errors
- Normalize JSON into a `pandas` DataFrame
- Practice with short exercises


## 0. Setup
Most Codespaces images already include `requests`. If you need it, run the install cell once.


In [1]:
# Optional: run once if needed
# %pip install requests pandas

import requests
import json
import pandas as pd


## 1. What is an API?
**API** (Application Programming Interface) endpoints let software exchange data.

Most modern APIs return data as **JSON** (JavaScript Object Notation), which maps cleanly to Python:
- JSON object → Python `dict`
- JSON array → Python `list`


## 2. Your first API call (GitHub)
We'll call GitHub’s public API to fetch metadata about the class repo.


In [2]:
import requests
import json
import pandas as pd


In [3]:
url = "https://api.github.com/repos/Health-informatics-Data-Analytics/hi5304-notebooks"

r = requests.get(url, headers={"User-Agent": "hi5304"})
r.raise_for_status()  # raises an error for 4xx/5xx responses

data = r.json()  # JSON -> Python dict
type(data), list(data.keys())[:10]


(dict,
 ['id',
  'node_id',
  'name',
  'full_name',
  'private',
  'owner',
  'html_url',
  'description',
  'fork',
  'url'])

### Inspect a few fields


In [4]:
data["full_name"], data["open_issues_count"], data["stargazers_count"], data["updated_at"]


('Health-informatics-Data-Analytics/hi5304-notebooks',
 0,
 1,
 '2026-02-07T20:58:50Z')

### Pretty-print JSON (first 1200 characters)
This helps you *see* nesting and field names.


In [5]:
print(json.dumps(data, indent=2)[:1200])


{
  "id": 1129982692,
  "node_id": "R_kgDOQ1oq5A",
  "name": "hi5304-notebooks",
  "full_name": "Health-informatics-Data-Analytics/hi5304-notebooks",
  "private": false,
  "owner": {
    "login": "Health-informatics-Data-Analytics",
    "id": 253527654,
    "node_id": "O_kgDODxyGZg",
    "avatar_url": "https://avatars.githubusercontent.com/u/253527654?v=4",
    "gravatar_id": "",
    "url": "https://api.github.com/users/Health-informatics-Data-Analytics",
    "html_url": "https://github.com/Health-informatics-Data-Analytics",
    "followers_url": "https://api.github.com/users/Health-informatics-Data-Analytics/followers",
    "following_url": "https://api.github.com/users/Health-informatics-Data-Analytics/following{/other_user}",
    "gists_url": "https://api.github.com/users/Health-informatics-Data-Analytics/gists{/gist_id}",
    "starred_url": "https://api.github.com/users/Health-informatics-Data-Analytics/starred{/owner}{/repo}",
    "subscriptions_url": "https://api.github.com/users

## 3. Nested JSON
Many APIs (including FHIR) use nested objects.
Here, `owner` is a nested dictionary.


In [6]:
type(data["owner"]), data["owner"]["login"], data["owner"]["html_url"]


(dict,
 'Health-informatics-Data-Analytics',
 'https://github.com/Health-informatics-Data-Analytics')

### Safer access with `.get()`
APIs can change. `.get()` avoids `KeyError` if a field is missing.


In [7]:
license_info = data.get("license")  # could be None
language = data.get("language", "Not specified")

license_info, language


(None, 'Jupyter Notebook')

## 4. Common request components
- **URL**: the endpoint
- **Query parameters**: e.g., `?per_page=10`
- **Headers**: metadata (often auth + content type)
- **Timeouts**: prevent hanging forever


### Example: parameters (`params=`)
Fetch the 10 most recent issues (if any).


In [8]:
issues_url = "https://api.github.com/repos/Health-informatics-Data-Analytics/hi5304-notebooks/issues"

r = requests.get(
    issues_url,
    params={"per_page": 10, "state": "open"},
    headers={"User-Agent": "hi5304"},
    timeout=10
)
r.raise_for_status()
issues = r.json()

type(issues), len(issues)


(list, 0)

### Normalize a list of JSON objects into a DataFrame
If there are no open issues, the DataFrame will just be empty.


In [9]:
df_issues = pd.json_normalize(issues)
df_issues.head()


""


## 5. Error handling patterns (teachable + practical)
Students should learn how to interpret errors and respond safely.


In [10]:
def safe_get(url: str, *, params=None, headers=None, timeout=10):
    try:
        resp = requests.get(url, params=params, headers=headers, timeout=timeout)
        resp.raise_for_status()
        return resp
    except requests.exceptions.Timeout:
        print("⏱️ Timeout: the server took too long to respond.")
    except requests.exceptions.HTTPError as e:
        print(f"🚫 HTTP error: {e} | status={getattr(resp, 'status_code', None)}")
        try:
            print("Response body (first 300 chars):", resp.text[:300])
        except Exception:
            pass
    except requests.exceptions.RequestException as e:
        print("❗ Network/request error:", e)

resp = safe_get("https://api.github.com/rate_limit", headers={"User-Agent":"hi5304"})
rate_data = resp.json() if resp else None
rate_data and rate_data["rate"]


{'limit': 60,
 'remaining': 46,
 'reset': 1771038825,
 'used': 14,
 'resource': 'core'}

## 6. Rate limits (important for real-world APIs)
Many APIs limit requests per hour/day. GitHub exposes rate limit information.


In [11]:
if rate_data:
    core = rate_data["resources"]["core"]
    print("Limit:", core["limit"])
    print("Remaining:", core["remaining"])
    print("Resets (unix time):", core["reset"])


Limit: 60
Remaining: 46
Resets (unix time): 1771038825


## 7. API keys & secrets (DO NOT commit keys)
For course work, the safest pattern is:
- Store keys in **Codespaces Secrets** or an untracked `.env`
- Read them via environment variables

Example (won't print the full key):


In [12]:
import os

# Example only: set a Codespaces secret like OPENWEATHER_API_KEY (do not hardcode keys)
api_key = os.getenv("OPENWEATHER_API_KEY")

if api_key:
    print("✅ API key loaded (first 4 chars):", api_key[:4] + "...")
else:
    print("ℹ️ No API key found. That's okay for this notebook's public endpoints.")


ℹ️ No API key found. That's okay for this notebook's public endpoints.


## 8. Mini-project: pull commit activity and summarize
We'll fetch recent commits and build a tidy table.


In [13]:
commits_url = "https://api.github.com/repos/Health-informatics-Data-Analytics/hi5304-notebooks/commits"

resp = safe_get(commits_url, params={"per_page": 15}, headers={"User-Agent":"hi5304"})
commits = resp.json() if resp else []

rows = []
for c in commits:
    rows.append({
        "sha": c.get("sha", "")[:10],
        "author": (c.get("commit", {}).get("author", {}) or {}).get("name"),
        "date": (c.get("commit", {}).get("author", {}) or {}).get("date"),
        "message": (c.get("commit", {}) or {}).get("message", "").splitlines()[0][:80],
    })

df_commits = pd.DataFrame(rows)
df_commits.head(10)


,sha,author,date,message
0,cdbc6fe8e1,Patrick Dunn,2026-02-07T20:58:42Z,R
1,408fd6b90d,Patrick Dunn,2026-02-07T19:47:05Z,updates
2,f948ee94e4,Patrick Dunn,2026-02-07T19:40:19Z,updates
3,8559c2163e,Patrick Dunn,2026-02-07T19:23:47Z,update
4,a7be88a7c0,Patrick Dunn,2026-01-30T21:43:48Z,update
5,4c2353d738,Patrick Dunn,2026-01-30T21:39:25Z,R in jupyter and terminal
6,d5c04785c8,Patrick Dunn,2026-01-28T03:38:20Z,prevent
7,4db617e2b1,Patrick Dunn,2026-01-27T14:50:25Z,risk calc
8,7a51170466,Patrick Dunn,2026-01-27T14:49:51Z,new datasets
9,9113efdc36,Patrick Dunn,2026-01-25T19:39:19Z,combined calculator


### Quick summary


In [14]:
df_commits["author"].value_counts(dropna=False).head(10)


author
Patrick Dunn    15
Name: count, dtype: int64

## 9. Exercises (for students)
1) Change `per_page` from 15 to 5 in the commits request. What changes?
2) Add a column called `message_len` with the length of each commit message.
3) Using the `issues` endpoint, request `state=all` and find how many issues are returned.
4) (Challenge) Create a function that takes an owner + repo name and returns a DataFrame summary.


In [15]:
# Exercise 2 starter: add message_len
df_commits["message_len"] = df_commits["message"].fillna("").str.len()
df_commits[["sha","author","date","message_len","message"]].head(10)


,sha,author,date,message_len,message
0,cdbc6fe8e1,Patrick Dunn,2026-02-07T20:58:42Z,1,R
1,408fd6b90d,Patrick Dunn,2026-02-07T19:47:05Z,7,updates
2,f948ee94e4,Patrick Dunn,2026-02-07T19:40:19Z,7,updates
3,8559c2163e,Patrick Dunn,2026-02-07T19:23:47Z,6,update
4,a7be88a7c0,Patrick Dunn,2026-01-30T21:43:48Z,6,update
5,4c2353d738,Patrick Dunn,2026-01-30T21:39:25Z,25,R in jupyter and terminal
6,d5c04785c8,Patrick Dunn,2026-01-28T03:38:20Z,7,prevent
7,4db617e2b1,Patrick Dunn,2026-01-27T14:50:25Z,9,risk calc
8,7a51170466,Patrick Dunn,2026-01-27T14:49:51Z,12,new datasets
9,9113efdc36,Patrick Dunn,2026-01-25T19:39:19Z,19,combined calculator


## 10. Extension: swap in a health-focused API
Once this pattern is comfortable, we can repeat the same workflow with public health APIs like:
- CDC public datasets (Socrata)
- OpenFDA
- U.S. Census

Tell me which one you want, and I’ll provide a drop-in section.


Example: OpenFDA Drug Adverse Events API
What this API represents (context for students)

OpenFDA provides access to FDA Adverse Event Reporting System (FAERS) data:

Reports of side effects, medication errors, and adverse reactions

Used in pharmacovigilance and post-market surveillance

Real-world, messy, high-volume public health data

🔗 Conceptually similar to:

Disease surveillance feeds

Safety signal monitoring systems

Real-world evidence pipelines

Step 1: Define the endpoint

This query asks:

“Give me the 10 most recent adverse event reports involving metformin.”

In [16]:
import requests
import json
import pandas as pd

url = "https://api.fda.gov/drug/event.json"

params = {
    "search": "patient.drug.medicinalproduct:metformin",
    "limit": 10
}

r = requests.get(url, params=params, timeout=10)
r.raise_for_status()

data = r.json()
type(data), data.keys()


(dict, dict_keys(['meta', 'results']))

Step 2: Inspect the structure

In [17]:
print(json.dumps(data, indent=2)[:1500])


{
  "meta": {
    "disclaimer": "Do not rely on openFDA to make decisions regarding medical care. While we make every effort to ensure that data is accurate, you should assume all results are unvalidated. We may limit or otherwise restrict your access to the API in line with our Terms of Service.",
    "terms": "https://open.fda.gov/terms/",
    "license": "https://open.fda.gov/license/",
    "last_updated": "2026-01-27",
    "results": {
      "skip": 0,
      "limit": 10,
      "total": 419462
    }
  },
  "results": [
    {
      "safetyreportversion": "2",
      "safetyreportid": "10003641",
      "primarysourcecountry": "US",
      "occurcountry": "US",
      "transmissiondateformat": "102",
      "transmissiondate": "20150720",
      "reporttype": "1",
      "serious": "2",
      "receivedateformat": "102",
      "receivedate": "20140312",
      "receiptdateformat": "102",
      "receiptdate": "20150312",
      "fulfillexpeditecriteria": "2",
      "companynumb": "US-ABBVIE-13P-1

Step 3: Extract the results

In [18]:
events = data["results"]
len(events), type(events)


(10, list)

Step 4: Normalize JSON into a DataFrame

In [19]:
rows = []

for e in events:
    patient = e.get("patient", {})
    drugs = patient.get("drug", [])
    reactions = patient.get("reaction", [])

    rows.append({
        "safety_report_id": e.get("safetyreportid"),
        "received_date": e.get("receivedate"),
        "serious": e.get("serious"),
        "drug_count": len(drugs),
        "reaction_count": len(reactions),
        "reactions": "; ".join(
            [r.get("reactionmeddrapt", "") for r in reactions[:3]]
        )
    })

df = pd.DataFrame(rows)
df


,safety_report_id,received_date,serious,drug_count,reaction_count,reactions
0,10003641,20140312,2,6,1,Medication residue present
1,10003642,20140312,1,11,1,Osteomyelitis
2,10003647,20140312,1,6,2,Erysipelas; Sepsis
3,10003803,20140312,2,4,3,Balance disorder; Gait disturbance; Gait distu...
4,10003915,20140312,2,11,1,Weight increased
5,10003959,20140312,2,11,3,Splenomegaly; Dyspnoea; Insomnia
6,10003993,20140312,1,9,2,Metastases to bone; Osteonecrosis of jaw
7,10003998,20140312,2,15,11,Feeling abnormal; Pneumonia; Suicidal ideation
8,10004006,20140312,2,10,5,Nausea; Vomiting; Abdominal pain upper
9,10004017,20140312,2,6,1,Epistaxis


CDC (Socrata): Population health / surveillance-style indicators

Example dataset: CDC U.S. Chronic Disease Indicators (Socrata dataset ID: hksd-2xuw).

In [20]:
import requests
import pandas as pd

# Socrata SODA endpoint pattern:
# https://data.cdc.gov/resource/<dataset_id>.json
url = "https://data.cdc.gov/resource/hksd-2xuw.json"

params = {
    "$select": "yearstart,locationdesc,topic,question,datavalueunit,datavalue",
    "$where": "yearstart >= 2020 AND locationdesc = 'Texas'",
    "$limit": 25
}

r = requests.get(url, params=params, timeout=20)
r.raise_for_status()
rows = r.json()

df = pd.DataFrame(rows)
df.head(10)


HTTPError: 500 Server Error: Server Error for url: https://data.cdc.gov/resource/hksd-2xuw.json?%24select=yearstart%2Clocationdesc%2Ctopic%2Cquestion%2Cdatavalueunit%2Cdatavalue&%24where=yearstart+%3E%3D+2020+AND+locationdesc+%3D+%27Texas%27&%24limit=25

U.S. Census API: SDOH / demographics (ACS)

Example: ACS 5-year (total population variable B01001_001E) using the Census “examples” endpoint for 2022.

In [ ]:
import requests
import pandas as pd
import os

# Census API base (ACS 5-year, 2022 example)
base = "https://api.census.gov/data/2022/acs/acs5"

# Optional: Census API key (recommended). If you don't have one, this may still work but can be rate-limited.
key = os.getenv("CENSUS_API_KEY")

params = {
    "get": "NAME,B01001_001E",   # NAME + total population estimate
    "for": "state:*"            # all states
}
if key:
    params["key"] = key

r = requests.get(base, params=params, timeout=20)
r.raise_for_status()
data = r.json()

# First row is headers
cols = data[0]
rows = data[1:]

df = pd.DataFrame(rows, columns=cols)
df["B01001_001E"] = pd.to_numeric(df["B01001_001E"], errors="coerce")
df.sort_values("B01001_001E", ascending=False).head(10)


CMS Data API: Utilization / cost / quality measures (data.cms.gov)

CMS provides dataset-specific API docs pages that include the dataset UUID(s) you query via /dataset/{uuid}/data.

Example dataset: Medicare Inpatient Hospitals – by Provider
Use the 2023 version UUID shown on the CMS API docs page: ad17abc4-0e93-4828-bba5-e8f39ddafbdb.

In [ ]:
import requests
import pandas as pd

uuid_2023 = "ad17abc4-0e93-4828-bba5-e8f39ddafbdb"
url = f"https://data.cms.gov/data-api/v1/dataset/{uuid_2023}/data"

params = {
    "limit": 25,
    # You can filter with "query" for simple text matching.
    # For structured filtering/column selection, check the dataset's api-docs page.
    "query": "Texas"
}

r = requests.get(url, params=params, timeout=30)
r.raise_for_status()
payload = r.json()

# CMS returns objects; often the records are in payload (list) or payload['data'] depending on endpoint/version
rows = payload if isinstance(payload, list) else payload.get("data", payload)

df = pd.DataFrame(rows)
df.head(10)
